# Crear muestras de entrenamiento con validación global reservada

`TAMANO_MUESTRA` define el total de filas/imágenes de **train**. Este notebook no
divide la muestra ni crea un conjunto de validación. Todas las muestras comparten
la reserva fija de `datasets/samples/data/validation.parquet`, tomada de la muestra
original de 20.000 imágenes. No se modifica ese archivo.

Antes de calcular cupos se excluyen las filas cuyo `image_id` **o** `panoid` aparece
en la validación global: tampoco se incorporan otras vistas del mismo panorama.
Las filas con identificadores faltantes o vacíos no son elegibles. La reserva debe
existir, tener ambas columnas y contener identificadores válidos; de lo contrario,
la ejecución se detiene. Solo se leen sus IDs, sin cargar las imágenes en memoria.

`MIN_FILAS_PAIS` filtra los países según `assets/countries_count.csv`; no exige ese
mínimo en la muestra final. Con `BALANCEADO=True` los cupos por país difieren a lo
sumo en una imagen; con `False` son proporcionales a las filas disponibles **después
de excluir la reserva**. Si no alcanza la disponibilidad, se informa el problema.
La selección es sin reemplazo y reproducible con `SEMILLA`.

Se conservan todas las columnas e imágenes sin decodificarlas. La salida incluye
solo `data/train.parquet`, índices, resumen por país y configuración con la ruta y
huella de los IDs reservados. Las muestras anteriores no se sobrescriben ni se
filtran retroactivamente: deben regenerarse para aplicar esta reserva.

Como mejora posterior, un script separado podrá crear la validación global y
guardar sus `image_id` y `panoid`. La reserva se crea una vez y se mantiene fija;
el muestreo de train solo la consulta.


In [1]:
from pathlib import Path
import json
import hashlib
import os

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display
from tqdm.auto import tqdm

# Parámetros editables.
TAMANO_MUESTRA = 50_000  # Cantidad de imágenes de train, sin la reserva global.
MIN_FILAS_PAIS = 1000  # Umbral sobre cantidad en el CSV, no sobre la muestra.
CSV_CONTEOS = "src/notebooks/preprocessing/assets/countries_count.csv"
BALANCEADO = True  # True: cupos iguales (+/- 1); False: proporcionales.
VALIDACION_GLOBAL = "datasets/samples/data/validation.parquet"
SEMILLA = 42
PATRON_ARCHIVOS = "**/train-*.parquet"  # Solo train; usar "**/*.parquet" para todos los splits.
CARPETA_SALIDA = "datasets/samples"  # Relativa a la raíz del proyecto, o absoluta.
TAMANO_LOTE = 128  # Limita la memoria al leer imágenes.

c:\Users\jurad\Documents\facultad\rrnn2\tp-final\geo-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Localizar el dataset y el CSV

`DATA_URL` se toma del entorno si ya esta definida; de lo contrario se lee del `.env`. Las rutas relativas del dataset, del CSV y de la salida se resuelven respecto de la carpeta del `.env`. El CSV decide que paises participan; el patron de archivos decide de que Parquet se extraen sus filas.

In [2]:
def entero_valido(valor, nombre, minimo):
    if isinstance(valor, bool) or not isinstance(valor, (int, np.integer)) or valor < minimo:
        raise ValueError(f"{nombre} debe ser un entero >= {minimo}.")

for nombre in ("TAMANO_MUESTRA", "MIN_FILAS_PAIS", "TAMANO_LOTE"):
    entero_valido(globals()[nombre], nombre, 1)
entero_valido(SEMILLA, "SEMILLA", 0)
if not isinstance(BALANCEADO, bool):
    raise ValueError("BALANCEADO debe ser True o False.")

env_path = next(
    (carpeta / ".env" for carpeta in (Path.cwd(), *Path.cwd().parents)
     if (carpeta / ".env").is_file()),
    None,
)
if env_path is None:
    raise FileNotFoundError("No se encontró el .env del proyecto desde el directorio actual.")

valor_env = ""
for linea in env_path.read_text(encoding="utf-8-sig").splitlines():
    clave, separador, valor = linea.partition("=")
    if separador and clave.strip() == "DATA_URL":
        valor_env = valor.strip().strip("\"'")

ruta_configurada = os.environ.get("DATA_URL", valor_env).strip()
if not ruta_configurada:
    raise ValueError("Configurá DATA_URL en el .env con la carpeta del dataset local.")

raiz_proyecto = env_path.parent
ruta_datos = Path(ruta_configurada).expanduser()
if not ruta_datos.is_absolute():
    ruta_datos = raiz_proyecto / ruta_datos
ruta_datos = ruta_datos.resolve()
if not ruta_datos.is_dir():
    raise NotADirectoryError(f"DATA_URL apunta a una carpeta inexistente: {ruta_datos}")

archivos = sorted(p for p in ruta_datos.glob(PATRON_ARCHIVOS) if p.is_file())
if not archivos:
    raise FileNotFoundError(f"No se encontraron Parquet con {PATRON_ARCHIVOS!r} en {ruta_datos}")

raiz_salida = Path(CARPETA_SALIDA).expanduser()
if not raiz_salida.is_absolute():
    raiz_salida = raiz_proyecto / raiz_salida
raiz_salida = raiz_salida.resolve()
# Evitar incorporar muestras previas como fuente en ejecuciones posteriores.
if raiz_salida == ruta_datos or ruta_datos in raiz_salida.parents:
    raise ValueError("CARPETA_SALIDA debe estar fuera de la carpeta del dataset de entrada.")

print(f"Dataset: {ruta_datos}")
print(f"Archivos de entrada: {len(archivos):,}")

ruta_csv = Path(CSV_CONTEOS).expanduser()
if not ruta_csv.is_absolute():
    ruta_csv = raiz_proyecto / ruta_csv
ruta_csv = ruta_csv.resolve()
# Preservar NA (Namibia): pandas lo interpreta como faltante por defecto.
conteos_csv = pd.read_csv(ruta_csv, dtype={"pais": "string"}, keep_default_na=False)
if not {"pais", "cantidad"}.issubset(conteos_csv.columns):
    raise ValueError("El CSV debe contener las columnas pais y cantidad.")
conteos_csv = conteos_csv[["pais", "cantidad"]].copy()
conteos_csv["pais"] = conteos_csv["pais"].str.strip().str.upper().replace("", pd.NA)
conteos_csv["cantidad"] = pd.to_numeric(conteos_csv["cantidad"], errors="raise")
if (conteos_csv.isna().any().any() or (conteos_csv["cantidad"] < 0).any()
        or (conteos_csv["cantidad"] % 1 != 0).any()):
    raise ValueError("El CSV debe tener codigos presentes y cantidades enteras no negativas.")
# TR figura dos veces con el mismo conteo: no sumar ni duplicar ese pais.
conteos_csv = conteos_csv.drop_duplicates()
if conteos_csv["pais"].duplicated().any():
    raise ValueError("El CSV tiene conteos contradictorios para un mismo pais.")
conteos_csv = conteos_csv.set_index("pais")["cantidad"].astype("int64").sort_index()
paises_admitidos = conteos_csv.index[conteos_csv >= MIN_FILAS_PAIS]
if paises_admitidos.empty:
    raise ValueError("Ningun pais del CSV alcanza MIN_FILAS_PAIS.")
print(f"Paises admitidos por el CSV: {len(paises_admitidos)}")

Dataset: C:\Users\jurad\Documents\facultad\rrnn2\tp-final\world-streetview-500k\data
Archivos de entrada: 393
Paises admitidos por el CSV: 71


## Leer los identificadores de la validación global

Se reserva por `image_id` y por `panoid`, incluso si la fuente contiene otras vistas
que no figuran como imágenes individuales en validación. El filtro se aplica antes
del muestreo. La huella identifica la reserva usada sin recorrer los bytes de imágenes.


In [3]:
ruta_validacion_global = Path(VALIDACION_GLOBAL).expanduser()
if not ruta_validacion_global.is_absolute():
    ruta_validacion_global = raiz_proyecto / ruta_validacion_global
ruta_validacion_global = ruta_validacion_global.resolve()
if not ruta_validacion_global.is_file():
    raise FileNotFoundError(f"Falta la validación global: {ruta_validacion_global}")
estado_validacion = ruta_validacion_global.stat()
with pq.ParquetFile(ruta_validacion_global) as parquet:
    if not {"image_id", "panoid"}.issubset(parquet.schema_arrow.names):
        raise ValueError("La validación global debe tener image_id y panoid.")
    ids_validacion = parquet.read(columns=["image_id", "panoid"]).to_pandas()
for columna in ids_validacion:
    ids_validacion[columna] = ids_validacion[columna].astype("string").str.strip().replace("", pd.NA)
if ids_validacion.empty or ids_validacion.isna().any().any():
    raise ValueError("La validación global debe tener filas e identificadores no vacíos.")
if ids_validacion["image_id"].duplicated().any():
    raise ValueError("La validación global contiene image_id duplicados.")
ids_ordenados = ids_validacion.sort_values(["image_id", "panoid"]).to_numpy().tolist()
huella_ids = hashlib.sha256(json.dumps(ids_ordenados, ensure_ascii=False, separators=(",", ":")).encode("utf-8")).hexdigest()
info_validacion = {
    "ruta": str(ruta_validacion_global),
    "filas": len(ids_validacion),
    "panoramas": int(ids_validacion["panoid"].nunique()),
    "columnas_exclusion": ["image_id", "panoid"],
    "sha256_ids": huella_ids,
    "bytes": estado_validacion.st_size,
    "mtime_ns": estado_validacion.st_mtime_ns,
}
print(f"Validación global: {len(ids_validacion):,} imágenes; {info_validacion['panoramas']:,} panoramas reservados.")


Validación global: 3,982 imágenes; 3,685 panoramas reservados.


## Calcular cupos después de excluir la validación global

La primera pasada lee únicamente `country_code`, `image_id` y `panoid`. Se mantiene
el índice global sobre los archivos ordenados para copiar luego las filas elegidas
por lotes. El resumen distingue disponibilidad original, filas reservadas por ID o
panorama, filas con identificadores inválidos y disponibilidad efectiva para train.
El umbral del CSV se aplica igual que antes; los cupos usan la disponibilidad efectiva.


In [4]:
partes = []
inventario = []
esquema = None
for archivo in tqdm(archivos, desc="Leyendo países e identificadores"):
    with pq.ParquetFile(archivo) as parquet:
        if not {"country_code", "image_id", "panoid"}.issubset(parquet.schema_arrow.names):
            raise ValueError(f"Falta country_code, image_id o panoid en {archivo}")
        if esquema is None:
            esquema = parquet.schema_arrow
        elif not esquema.equals(parquet.schema_arrow, check_metadata=True):
            raise ValueError(f"El esquema difiere del primer archivo: {archivo}")
        partes.append(parquet.read(columns=["country_code", "image_id", "panoid"]).to_pandas())
        inventario.append({
            "archivo": str(archivo), "filas": parquet.metadata.num_rows,
            "bytes": archivo.stat().st_size, "mtime_ns": archivo.stat().st_mtime_ns,
        })

metadatos = pd.concat(partes, ignore_index=True)
del partes
paises = metadatos["country_code"].astype("string").str.strip().str.upper().replace("", pd.NA)
image_ids = metadatos["image_id"].astype("string").str.strip().replace("", pd.NA)
panoids = metadatos["panoid"].astype("string").str.strip().replace("", pd.NA)
del metadatos
ids_invalidos = image_ids.isna() | panoids.isna()
reservadas = image_ids.isin(ids_validacion["image_id"]) | panoids.isin(ids_validacion["panoid"])
disponibles_train = ~ids_invalidos & ~reservadas
conteos_originales = paises.value_counts().sort_index()
conteos = paises.loc[disponibles_train].value_counts().sort_index()
conteos.index.name = "country_code"
print(f"Filas totales: {len(paises):,}; sin país: {paises.isna().sum():,}")
print(f"Excluidas por validación global (image_id o panoid): {reservadas.sum():,}")
print(f"Identificadores inválidos: {ids_invalidos.sum():,}; disponibles para train: {disponibles_train.sum():,}")


def asignar_cupos(conteos, tamano, paises_admitidos, balanceado=False):
    """Cupos iguales (+/- 1) o proporcionales, sin reemplazo."""
    elegibles = conteos[conteos.index.isin(paises_admitidos) & (conteos > 0)].sort_index().astype("int64")
    disponible = int(elegibles.sum())
    if disponible == 0:
        raise ValueError("No hay filas locales de los paises admitidos por el CSV.")
    if tamano > disponible:
        raise ValueError(f"Se pidieron {tamano:,} filas pero solo hay {disponible:,} elegibles.")
    if balanceado:
        base, resto = divmod(tamano, len(elegibles))
        pueden_recibir_extra = elegibles.index[elegibles > base]
        if (elegibles < base).any() or len(pueden_recibir_extra) < resto:
            minimo_local = int(elegibles.min())
            maximo_balanceado = minimo_local * len(elegibles) + int((elegibles > minimo_local).sum())
            raise ValueError(
                f"No hay filas suficientes para balancear {tamano:,} muestras sin reemplazo. "
                f"El maximo balanceado con estos paises es {maximo_balanceado:,}. "
                "Reduci TAMANO_MUESTRA o usa BALANCEADO = False."
            )
        cupos = pd.Series(base, index=elegibles.index, dtype="int64")
        # Empates resueltos por codigo de pais para un resultado reproducible.
        cupos.loc[pueden_recibir_extra[:resto]] += 1
        return cupos[cupos > 0]
    numeradores = elegibles * tamano
    cupos = numeradores // disponible
    faltantes = tamano - int(cupos.sum())
    orden = (numeradores % disponible).sort_values(ascending=False, kind="stable")
    cupos.loc[orden.index[:faltantes]] += 1
    return cupos[cupos > 0]



cupos = asignar_cupos(conteos, TAMANO_MUESTRA, paises_admitidos, BALANCEADO)
modo_muestreo = "balanceado" if BALANCEADO else "proporcional"
print(f"Muestreo: {modo_muestreo}")
resumen = pd.concat([
    conteos_csv.rename("cantidad_csv"),
    conteos_originales.rename("disponibles_originales"),
    paises.loc[reservadas].value_counts().rename("excluidas_validacion_global"),
    paises.loc[ids_invalidos & ~reservadas].value_counts().rename("excluidas_ids_invalidos"),
    conteos.rename("disponibles"),
], axis=1).sort_index()
resumen.index.name = "country_code"
columnas_conteos = ["disponibles_originales", "excluidas_validacion_global",
                    "excluidas_ids_invalidos", "disponibles"]
resumen[columnas_conteos] = resumen[columnas_conteos].fillna(0).astype("int64")
assert (resumen["disponibles_originales"] == resumen["excluidas_validacion_global"]
        + resumen["excluidas_ids_invalidos"] + resumen["disponibles"]).all()
resumen["admitido_csv"] = resumen.index.isin(paises_admitidos)
resumen["seleccionadas"] = cupos.reindex(resumen.index, fill_value=0)
resumen["incluido"] = resumen["seleccionadas"] > 0
resumen["motivo"] = np.select(
    [resumen["cantidad_csv"].isna(), ~resumen["admitido_csv"],
     resumen["disponibles"] == 0, ~resumen["incluido"]],
    ["ausente del CSV", "cantidad CSV menor al umbral", "sin filas elegibles para train", "cupo cero"],
    default="incluido",
)
display(resumen)
print(f"Países representados en train: {len(cupos)}; filas de salida: {cupos.sum():,}")


Leyendo países e identificadores: 100%|██████████| 393/393 [00:02<00:00, 152.55it/s]


Filas totales: 1,999,060; sin país: 7,932
Excluidas por validación global (image_id o panoid): 14,740
Identificadores inválidos: 0; disponibles para train: 1,984,320
Muestreo: balanceado


,cantidad_csv,disponibles_originales,excluidas_validacion_global,excluidas_ids_invalidos,disponibles,admitido_csv,seleccionadas,incluido,motivo
country_code,,,,,,,,,
AD,24,24,0,0,24,False,0,False,cantidad CSV menor al umbral
AE,780,780,0,0,780,False,0,False,cantidad CSV menor al umbral
AL,624,624,0,0,624,False,0,False,cantidad CSV menor al umbral
AR,29668,29668,224,0,29444,True,705,True,incluido
AT,9096,9096,196,0,8900,True,705,True,incluido
...,...,...,...,...,...,...,...,...,...
US,476840,476840,224,0,476616,True,704,True,incluido
UY,3000,3000,196,0,2804,True,704,True,incluido
VI,44,44,0,0,44,False,0,False,cantidad CSV menor al umbral


Países representados en train: 71; filas de salida: 50,000


In [5]:
rng = np.random.default_rng(SEMILLA)
# Conservar los índices de origen; no renumerar después de excluir la reserva.
grupos = paises.loc[disponibles_train].groupby(paises.loc[disponibles_train], sort=True).groups
indices_train = np.sort(np.concatenate([
    rng.choice(np.asarray(grupos[pais], dtype=np.int64), size=int(cantidad), replace=False)
    for pais, cantidad in cupos.items()
])).astype(np.int64)
del grupos
indices = indices_train  # Toda la muestra es train.

conteos_seleccionados = paises.iloc[indices_train].value_counts().sort_index()
assert len(indices_train) == TAMANO_MUESTRA
assert len(np.unique(indices_train)) == TAMANO_MUESTRA
assert conteos_seleccionados.to_dict() == cupos.to_dict()
assert disponibles_train.iloc[indices_train].all()
assert not image_ids.iloc[indices_train].isin(ids_validacion["image_id"]).any()
assert not panoids.iloc[indices_train].isin(ids_validacion["panoid"]).any()
seleccion = pd.DataFrame({"pais": paises.iloc[indices_train], "panoid": panoids.iloc[indices_train]})
if (seleccion.groupby("panoid")["pais"].nunique() > 1).any():
    raise ValueError("Hay panoramas asociados a varios países en la muestra. Revisá la fuente.")
resumen["train"] = conteos_seleccionados.reindex(resumen.index, fill_value=0).astype("int64")
assert resumen["train"].equals(resumen["seleccionadas"])
print(f"Train: {len(indices_train):,}; validación global externa: {len(ids_validacion):,}")
display(resumen)

nombre_muestra = (
    f"muestra_n-{TAMANO_MUESTRA}_umbral-csv-{MIN_FILAS_PAIS}"
    f"_{modo_muestreo}_paises-{len(cupos)}_seed-{SEMILLA}_solo-train"
)
ruta_salida = raiz_salida / nombre_muestra
if ruta_salida.exists():
    raise FileExistsError(f"La salida ya existe: {ruta_salida}. Cambiá la configuración o CARPETA_SALIDA.")
print(f"Salida: {ruta_salida}")


Train: 50,000; validación global externa: 3,982


,cantidad_csv,disponibles_originales,excluidas_validacion_global,excluidas_ids_invalidos,disponibles,admitido_csv,seleccionadas,incluido,motivo,train
country_code,,,,,,,,,,
AD,24,24,0,0,24,False,0,False,cantidad CSV menor al umbral,0
AE,780,780,0,0,780,False,0,False,cantidad CSV menor al umbral,0
AL,624,624,0,0,624,False,0,False,cantidad CSV menor al umbral,0
AR,29668,29668,224,0,29444,True,705,True,incluido,705
AT,9096,9096,196,0,8900,True,705,True,incluido,705
...,...,...,...,...,...,...,...,...,...,...
US,476840,476840,224,0,476616,True,704,True,incluido,704
UY,3000,3000,196,0,2804,True,704,True,incluido,704
VI,44,44,0,0,44,False,0,False,cantidad CSV menor al umbral,0


Salida: C:\Users\jurad\Documents\facultad\rrnn2\tp-final\geo-project\datasets\samples\muestra_n-50000_umbral-csv-1000_balanceado_paises-71_seed-42_solo-train


## Guardar solo train

Se crea únicamente `data/train.parquet`, conservando todas las columnas y el
esquema original. Se escribe por lotes en orden de origen. Se vuelve a comprobar
la ausencia de IDs y panoramas reservados sobre el Parquet guardado, antes de
crear `COMPLETO`. Si cambia la fuente o la reserva durante la ejecución, se detiene.

La carpeta incluye `resumen_paises.csv`, los índices globales seleccionados y
`config.json`, con la procedencia de la reserva y la huella SHA-256 de sus IDs.
No se sobrescriben salidas anteriores; una interrupción deja la carpeta incompleta.
`archivos_salida` declara solo train: para evaluar, los notebooks de modelado deben
usar por separado `datasets/samples/data/validation.parquet`.

Las imágenes embebidas se copian sin decodificarlas; las rutas externas, si existen,
no se convierten en bytes. Para cargar ambos conjuntos se puede usar
`load_dataset("parquet", data_files={"train": str(ruta_salida / "data/train.parquet"),
"validation": str(ruta_validacion_global)})`.


In [6]:
def comprobar_validacion_global():
    estado = ruta_validacion_global.stat()
    if (estado.st_size != info_validacion["bytes"]
            or estado.st_mtime_ns != info_validacion["mtime_ns"]):
        raise RuntimeError("La validación global cambió. Reejecutá desde el inicio.")


def guardar_muestra(archivos, inventario, indices_train, esquema, destino, tamano_lote):
    for archivo, info in zip(archivos, inventario, strict=True):
        estado = archivo.stat()
        if estado.st_size != info["bytes"] or estado.st_mtime_ns != info["mtime_ns"]:
            raise RuntimeError(f"La fuente cambió desde el conteo: {archivo}. Reejecutá desde el inicio.")
    comprobar_validacion_global()
    destino.mkdir(parents=True, exist_ok=False)
    (destino / "data").mkdir()
    guardadas = 0
    inicio_archivo = 0
    with pq.ParquetWriter(destino / "data/train.parquet", esquema, compression="snappy") as escritor:
        for archivo, info in tqdm(list(zip(archivos, inventario, strict=True)), desc="Guardando train"):
            fin_archivo = inicio_archivo + info["filas"]
            seleccion = indices_train[np.searchsorted(indices_train, inicio_archivo):np.searchsorted(indices_train, fin_archivo)]
            if len(seleccion):
                with pq.ParquetFile(archivo) as parquet:
                    inicio_grupo = inicio_archivo
                    for grupo in range(parquet.num_row_groups):
                        fin_grupo = inicio_grupo + parquet.metadata.row_group(grupo).num_rows
                        if np.searchsorted(seleccion, inicio_grupo) < np.searchsorted(seleccion, fin_grupo):
                            inicio_lote = inicio_grupo
                            for lote in parquet.iter_batches(batch_size=tamano_lote, row_groups=[grupo]):
                                fin_lote = inicio_lote + lote.num_rows
                                globales = seleccion[np.searchsorted(seleccion, inicio_lote):np.searchsorted(seleccion, fin_lote)]
                                if len(globales):
                                    posiciones = globales - inicio_lote
                                    tabla = pa.Table.from_batches([lote]).take(pa.array(posiciones))
                                    escritor.write_table(tabla)
                                    guardadas += len(posiciones)
                                inicio_lote = fin_lote
                        inicio_grupo = fin_grupo
            inicio_archivo = fin_archivo
    if guardadas != len(indices_train):
        raise RuntimeError(f"Filas guardadas: {guardadas}; se esperaban {len(indices_train)}.")
    return {"train": guardadas}


guardadas = guardar_muestra(
    archivos, inventario, indices_train, esquema, ruta_salida, TAMANO_LOTE
)
# Verificar lo escrito, no solo los índices seleccionados.
ids_guardados = pq.read_table(ruta_salida / "data/train.parquet", columns=["image_id", "panoid"]).to_pandas()
for columna in ("image_id", "panoid"):
    valores = ids_guardados[columna].astype("string").str.strip().replace("", pd.NA)
    if valores.isna().any() or valores.isin(ids_validacion[columna]).any():
        raise RuntimeError(f"Train contiene {columna} inválidos o reservados para validación.")
comprobar_validacion_global()
resumen.to_csv(ruta_salida / "resumen_paises.csv")
np.save(ruta_salida / "indices_seleccionados.npy", indices_train, allow_pickle=False)
np.save(ruta_salida / "indices_train.npy", indices_train, allow_pickle=False)
configuracion = {
    "tipo_muestra": "solo_train",
    "tamano_muestra": TAMANO_MUESTRA,
    "filas_train": len(indices_train),
    "archivos_salida": {"train": "data/train.parquet"},
    "validacion_global": info_validacion,
    "min_filas_pais_csv": MIN_FILAS_PAIS,
    "csv_conteos": str(ruta_csv),
    "conteos_csv": {pais: int(cantidad) for pais, cantidad in conteos_csv.items()},
    "paises_admitidos_csv": paises_admitidos.tolist(),
    "semilla": SEMILLA,
    "balanceado": BALANCEADO,
    "paises_incluidos": cupos.index.tolist(),
    "metodo": f"exclusión de validación global por image_id y panoid + muestreo {modo_muestreo}",
    "unidad": "fila/imagen",
    "ruta_datos": str(ruta_datos),
    "patron_archivos": PATRON_ARCHIVOS,
    "archivos_fuente": inventario,
    "filas_fuente": len(paises),
    "filas_sin_pais": int(paises.isna().sum()),
    "filas_excluidas_validacion_global": int(reservadas.sum()),
    "filas_ids_invalidos": int(ids_invalidos.sum()),
}
(ruta_salida / "config.json").write_text(
    json.dumps(configuracion, ensure_ascii=False, indent=2), encoding="utf-8"
)
(ruta_salida / "COMPLETO").write_text(f"{guardadas['train']} filas guardadas: {guardadas}.\n", encoding="utf-8")
print(f"Muestra guardada: {guardadas['train']:,} imágenes de train en {ruta_salida}")


Guardando train: 100%|██████████| 393/393 [11:47<00:00,  1.80s/it]


Muestra guardada: 50,000 imágenes de train en C:\Users\jurad\Documents\facultad\rrnn2\tp-final\geo-project\datasets\samples\muestra_n-50000_umbral-csv-1000_balanceado_paises-71_seed-42_solo-train
